In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans

In [2]:
# Load scaled patient features

df_scaled = pd.read_csv(
    "../outputs/processed_data/patient_features_scaled.csv"
)

print("Dataset Shape:", df_scaled.shape)

display(df_scaled.head())

Dataset Shape: (50000, 8)


,patient_id,Recency,Frequency,Monetary,Admission_Frequency,Average_LOS,Lab_Frequency,Seasonality
0,P000001,0.476281,0.108664,0.256826,-0.908192,-0.93317,-0.510987,0.626728
1,P000002,0.895072,-0.927572,0.014870,-0.908192,-0.93317,-0.510987,1.232359
2,P000003,-0.093747,0.945274,0.830264,2.336875,0.82465,1.439255,-0.341213
3,P000004,-0.974302,0.442257,0.547028,-0.908192,-0.93317,1.199281,0.220843
4,P000005,0.093481,-0.927572,0.472493,-0.908192,-0.93317,-0.510987,1.232359


In [3]:
# Select only the seven clustering features

feature_columns = [
    'Recency',
    'Frequency',
    'Monetary',
    'Admission_Frequency',
    'Average_LOS',
    'Lab_Frequency',
    'Seasonality'
]

X = df_scaled[feature_columns]

print("Clustering Features Shape:", X.shape)

Clustering Features Shape: (50000, 7)


In [5]:
# Set the optimal number of clusters

optimal_k = 4

print("Selected Number of Clusters:", optimal_k)

Selected Number of Clusters: 4


In [6]:
# Apply K-Means clustering

kmeans = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

# Assign cluster labels to patients

df_scaled["Cluster"] = kmeans.fit_predict(X)

print("K-Means Clustering Completed!")

print("\nCluster Distribution:")
print(df_scaled["Cluster"].value_counts().sort_index())

K-Means Clustering Completed!

Cluster Distribution:
Cluster
0     4713
1    20717
2     4723
3    19847
Name: count, dtype: int64


In [7]:
# Calculate the number and percentage of patients in each cluster

cluster_summary = df_scaled["Cluster"].value_counts().sort_index().reset_index()

cluster_summary.columns = ["Cluster", "Patient_Count"]

cluster_summary["Percentage"] = (
    cluster_summary["Patient_Count"] / len(df_scaled) * 100
)

display(cluster_summary.round(2))

,Cluster,Patient_Count,Percentage
0,0,4713,9.43
1,1,20717,41.43
2,2,4723,9.45
3,3,19847,39.69


In [8]:
# Save patient IDs with their cluster assignments

clustered_data = df_scaled[["patient_id", "Cluster"]].copy()

clustered_data.to_csv(
    "../outputs/clustered_patients.csv",
    index=False
)

print("Cluster assignments saved successfully!")

display(clustered_data.head())

Cluster assignments saved successfully!


,patient_id,Cluster
0,P000001,3
1,P000002,3
2,P000003,1
3,P000004,3
4,P000005,3


In [9]:
# Load original patient features

patient_features = pd.read_csv(
    "../outputs/patient_features.csv"
)

# Merge original features with cluster labels

clustered_patients = patient_features.merge(
    clustered_data,
    on="patient_id",
    how="inner"
)

print("Merged Dataset Shape:", clustered_patients.shape)

display(clustered_patients.head())

Merged Dataset Shape: (50000, 15)


,patient_id,home_hospital_id,gender,age,Recency,Frequency,Monetary,Admission_Frequency,Average_LOS,Lab_Frequency,Growth,Seasonality,No_Growth_Baseline,Has_OPD_Visit,Cluster
0,P000001,H004,Male,60,249.0,3.0,4957.45,0.0,0.0,1.0,-50.0,2.486326,False,1,3
1,P000002,H002,Male,29,433.0,1.0,2429.05,0.0,0.0,1.0,-100.0,3.464102,False,1,3
2,P000003,H003,Male,28,117.0,6.0,26877.75,5.0,4.0,6.0,-50.0,1.348400,False,1,1
3,P000004,H001,Female,35,36.0,4.0,11662.68,0.0,0.0,5.0,200.0,1.954017,False,1,3
4,P000005,H002,Male,34,150.0,1.0,9362.15,0.0,0.0,1.0,0.0,3.464102,True,1,3


In [10]:
# Save final clustered dataset with original features

clustered_patients.to_csv(
    "../outputs/clustered_patient_features.csv",
    index=False
)

print("Final clustered patient dataset saved successfully!")

Final clustered patient dataset saved successfully!
